# 02 — Feature Engineering

**Input:** `notebooks/data/raw_combined.csv` (684 rows, 93 cols)

**Output:** `notebooks/data/features_skaters.csv`, `notebooks/data/features_goalies.csv`

### What this notebook does
1. Cap-normalize the target variable (AAV as % of signing-year cap ceiling)
2. Convert icetime seconds to minutes/60 denominators for per-60 rates
3. Compute per-60 rates for all counting stats
4. Compute derived metrics: xGF%, Corsi%, HD save%, GSAA/60, faceoff%, zone start%, PP/SH time share
5. Add durability feature: `gp_pct` = games_played / 82
6. Handle nulls (fill pp/sh per-60 with 0 where icetime = 0 — player had no pp/pk time)
7. Build position-specific feature sets for F, D, G
8. Spot-check and audit before export

### Cap ceilings by season
| Season | Cap |
|--------|-----|
| 2019   | $81.5M |
| 2020   | $81.5M (COVID freeze) |
| 2021   | $81.5M (COVID freeze) |
| 2022   | $82.5M |
| 2023   | $83.5M |
| 2024   | $88.0M |

In [ ]:
import os
import pandas as pd
import numpy as np

NOTEBOOK_DIR = os.getcwd()
DATA_DIR     = os.path.join(NOTEBOOK_DIR, 'data')

df = pd.read_csv(os.path.join(DATA_DIR, 'raw_combined.csv'))
print(f'Loaded: {len(df):,} rows x {len(df.columns)} cols')
print(f'Positions: {df["position"].value_counts().to_dict()}')

## 1. Cap-Normalize the Target Variable

In [ ]:
# Signing-year cap ceilings (the cap in effect when the contract was signed)
CAP = {
    2019: 81_500_000,
    2020: 81_500_000,
    2021: 81_500_000,
    2022: 82_500_000,
    2023: 83_500_000,
    2024: 88_000_000,
    2025: 88_000_000,  # current cap (for model inference)
}

# signing_year = start_year (the year the contract began)
df['cap_at_signing'] = df['start_year'].map(CAP)
df['aav_pct_cap']    = df['aav'] / df['cap_at_signing']

print('AAV as % of cap ceiling:')
print(df['aav_pct_cap'].describe().round(4))
print()
print('Spot check:')
cols = ['player_name', 'start_year', 'aav', 'cap_at_signing', 'aav_pct_cap']
print(df[df['player_name'].isin(['Connor McDavid','Nathan MacKinnon','Cale Makar','Igor Shesterkin'])][cols].to_string())

## 2. Per-60 Rate Stats (Skaters)

MoneyPuck `icetime` is in **seconds**. To get per-60 minutes:
```
rate_per_60 = count / (icetime_seconds / 60)
```

In [ ]:
def per60(count_col, ice_col, df):
    """Rate per 60 minutes of icetime. Returns 0 where icetime is 0."""
    minutes = df[ice_col] / 60.0
    return np.where(minutes > 0, df[count_col] / minutes, 0.0)

# All-situations per-60 rates
df['goals_per60']         = per60('I_F_goals',           'icetime', df)
df['primaryA_per60']      = per60('I_F_primaryAssists',  'icetime', df)
df['points_per60']        = per60('I_F_goals',           'icetime', df) + per60('I_F_primaryAssists', 'icetime', df) + per60('I_F_secondaryAssists', 'icetime', df)
df['xG_per60']            = per60('I_F_xGoals',          'icetime', df)
df['shots_per60']         = per60('I_F_shotsOnGoal',     'icetime', df)
df['hdGoals_per60']       = per60('I_F_highDangerGoals', 'icetime', df)
df['hits_per60']          = per60('I_F_hits',            'icetime', df)
df['takeaways_per60']     = per60('I_F_takeaways',       'icetime', df)
df['giveaways_per60']     = per60('I_F_giveaways',       'icetime', df)
df['blocks_per60']        = per60('shotsBlockedByPlayer','icetime', df)
df['penDiff_per60']       = per60('penaltiesDrawn',      'icetime', df) - per60('penalties', 'icetime', df)

# Even-strength per-60 (ev_ prefix = 5on5 situation)
df['ev_goals_per60']      = per60('ev_I_F_goals',          'ev_icetime', df)
df['ev_primaryA_per60']   = per60('ev_I_F_primaryAssists', 'ev_icetime', df)
df['ev_xG_per60']         = per60('ev_I_F_xGoals',         'ev_icetime', df)

# Power-play per-60 (pp_ prefix = 5on4)
df['pp_goals_per60']      = per60('pp_I_F_goals',          'pp_icetime', df)
df['pp_primaryA_per60']   = per60('pp_I_F_primaryAssists', 'pp_icetime', df)
df['pp_xG_per60']         = per60('pp_I_F_xGoals',         'pp_icetime', df)

# Penalty-kill per-60 (sh_ prefix = 4on5)
df['pk_goals_per60']      = per60('sh_I_F_goals',          'sh_icetime', df)
df['pk_xG_per60']         = per60('sh_I_F_xGoals',         'sh_icetime', df)

print('Per-60 rates created. Sample (McDavid):')
rate_cols = ['player_name','goals_per60','primaryA_per60','points_per60','xG_per60','pp_goals_per60','ev_xG_per60']
print(df[df['player_name'] == 'Connor McDavid'][rate_cols].to_string())

## 3. On-Ice Percentage Metrics

In [ ]:
def safe_pct(num, den):
    """num / (num + den), returns 0.5 where denominator is 0."""
    total = num + den
    return np.where(total > 0, num / total, 0.5)

# xGF% — on-ice expected goals for percentage (all situations)
df['xGF_pct'] = safe_pct(df['OnIce_F_xGoals'], df['OnIce_A_xGoals'])

# Corsi% — all shot attempts for percentage
df['corsi_pct'] = safe_pct(df['OnIce_F_shotAttempts'], df['OnIce_A_shotAttempts'])

# EV xGF% (5on5 only — cleaner signal, no PP/PK skew)
df['ev_xGF_pct'] = safe_pct(df['ev_OnIce_F_xGoals'], df['ev_OnIce_A_xGoals'])

# EV Corsi%
df['ev_corsi_pct'] = safe_pct(df['ev_OnIce_F_shotAttempts'], df['ev_OnIce_A_shotAttempts'])

# Zone start % — fraction of shifts starting in the offensive zone
total_starts = df['I_F_oZoneShiftStarts'] + df['I_F_dZoneShiftStarts'] + df['I_F_neutralZoneShiftStarts']
df['ozone_start_pct'] = np.where(total_starts > 0, df['I_F_oZoneShiftStarts'] / total_starts, 0.5)

# Faceoff% (relevant mainly for centres; NaN for non-C positions is fine)
total_fo = df['faceoffsWon'] + df['faceoffsLost']
df['faceoff_pct'] = np.where(total_fo > 0, df['faceoffsWon'] / total_fo, np.nan)

# PP/SH icetime shares
df['pp_toi_share'] = np.where(df['icetime'] > 0, df['pp_icetime'] / df['icetime'], 0.0)
df['pk_toi_share'] = np.where(df['icetime'] > 0, df['sh_icetime'] / df['icetime'], 0.0)
df['ev_toi_share'] = np.where(df['icetime'] > 0, df['ev_icetime'] / df['icetime'], 0.0)

print('On-ice % metrics created.')
pct_cols = ['player_name','xGF_pct','ev_xGF_pct','corsi_pct','ozone_start_pct','faceoff_pct','pp_toi_share']
print(df[df['player_name'].isin(['Connor McDavid','Cale Makar','Nicklas Backstrom'])][pct_cols].to_string())

## 4. Durability & Contract Context

In [ ]:
# Durability: fraction of an 82-game season played
df['gp_pct'] = df['games_played'] / 82.0

# TOI per game (minutes) — proxy for usage/trust
df['toi_per_game'] = (df['icetime'] / 60.0) / df['games_played']

# Ensure age_at_signing is numeric
df['age_at_signing'] = pd.to_numeric(df['age_at_signing'], errors='coerce')

print('Durability features:')
dur_cols = ['player_name','games_played','gp_pct','toi_per_game','age_at_signing']
print(df[df['player_name'].isin(['Connor McDavid','Cale Makar','Auston Matthews'])][dur_cols].to_string())

## 5. Goalie Metrics

Derived from raw goalie stats: save%, HD save%, GSAA (goals saved above expected), GSAA/60.

In [ ]:
# Save % = (shots on goal - goals against) / shots on goal
df['save_pct'] = np.where(
    df['ongoal'].notna() & (df['ongoal'] > 0),
    (df['ongoal'] - df['goals']) / df['ongoal'],
    np.nan
)

# HD save % = (HD shots - HD goals) / HD shots
df['hd_save_pct'] = np.where(
    df['highDangerShots'].notna() & (df['highDangerShots'] > 0),
    (df['highDangerShots'] - df['highDangerGoals']) / df['highDangerShots'],
    np.nan
)

# GSAA = xGoals against - actual goals against (positive = better than expected)
df['gsaa'] = np.where(df['xGoals'].notna(), df['xGoals'] - df['goals'], np.nan)

# GSAA per 60 minutes
df['gsaa_per60'] = np.where(
    df['icetime'].notna() & (df['icetime'] > 0) & df['gsaa'].notna(),
    df['gsaa'] / (df['icetime'] / 60.0),
    np.nan
)

# Goals against average (per 60)
df['gaa'] = np.where(
    df['icetime'].notna() & (df['icetime'] > 0) & df['goals'].notna(),
    df['goals'] / (df['icetime'] / 60.0),
    np.nan
)

# Shots against per 60 (workload)
df['sa_per60'] = np.where(
    df['icetime'].notna() & (df['icetime'] > 0) & df['ongoal'].notna(),
    df['ongoal'] / (df['icetime'] / 60.0),
    np.nan
)

print('Goalie metrics:')
g_cols = ['player_name','save_pct','hd_save_pct','gsaa','gsaa_per60','gaa','games_played','aav']
goalies = df[df['position'] == 'G'].sort_values('aav', ascending=False)
print(goalies[g_cols].head(10).to_string())

## 6. Define Position-Specific Feature Sets

Different skills matter for different positions:
- **Forwards (F/C/LW/RW):** scoring, PP production, on-ice effects, faceoffs (C only)
- **Defensemen (D):** EV play, defensive zone, PP QB role, physical play
- **Goalies (G):** save metrics, HD save%, GSAA, workload

In [ ]:
# Shared features for all positions
SHARED = [
    'age_at_signing',
    'gp_pct',
    'toi_per_game',
]

# Forward features
FORWARD_FEATURES = SHARED + [
    # Production (all-situations)
    'goals_per60',
    'primaryA_per60',
    'points_per60',
    'xG_per60',
    'shots_per60',
    'hdGoals_per60',
    # EV production
    'ev_goals_per60',
    'ev_xG_per60',
    'ev_xGF_pct',
    'ev_corsi_pct',
    # PP
    'pp_goals_per60',
    'pp_xG_per60',
    'pp_toi_share',
    # Faceoffs (most useful for C, but included for all F — non-C will be near 0)
    'faceoff_pct',
    # Zone deployment
    'ozone_start_pct',
    # Physical / defensive
    'hits_per60',
    'takeaways_per60',
    'penDiff_per60',
]

# Defenseman features
DEFENSE_FEATURES = SHARED + [
    # Offensive contribution
    'primaryA_per60',
    'points_per60',
    'xG_per60',
    # EV two-way play
    'ev_xG_per60',
    'ev_xGF_pct',
    'ev_corsi_pct',
    # PP QB role
    'pp_primaryA_per60',
    'pp_xG_per60',
    'pp_toi_share',
    # PK role
    'pk_toi_share',
    # Physical / defensive
    'hits_per60',
    'blocks_per60',
    'takeaways_per60',
    'giveaways_per60',
    # Zone deployment
    'ozone_start_pct',
    'penDiff_per60',
]

# Goalie features
GOALIE_FEATURES = SHARED + [
    'save_pct',
    'hd_save_pct',
    'gsaa',
    'gsaa_per60',
    'gaa',
    'sa_per60',
]

TARGET = 'aav_pct_cap'

print(f'Forward features:  {len(FORWARD_FEATURES)}')
print(f'Defense features:  {len(DEFENSE_FEATURES)}')
print(f'Goalie features:   {len(GOALIE_FEATURES)}')

## 7. Build Position-Specific DataFrames

In [ ]:
FORWARD_POS = {'C', 'L', 'LW', 'R', 'RW', 'W', 'F'}
DEFENSE_POS = {'D', 'LD', 'RD'}

pos_upper = df['position'].str.upper()
is_forward = pos_upper.isin(FORWARD_POS)
is_defense = pos_upper.isin(DEFENSE_POS)
is_goalie  = pos_upper == 'G'

META = ['player_name', 'position', 'stat_season', 'start_year', 'aav', 'cap_at_signing']

def build_df(mask, features, label):
    sub = df[mask].copy()
    # Fill faceoff_pct NaN with 0.5 (neutral) for non-centre forwards
    if 'faceoff_pct' in features:
        sub['faceoff_pct'] = sub['faceoff_pct'].fillna(0.5)
    cols = META + [TARGET] + [f for f in features if f in sub.columns]
    sub = sub[cols].copy()
    null_count = sub[features].isnull().sum().sum()
    print(f'{label}: {len(sub):,} rows, {len(features)} features, {null_count} total nulls in feature matrix')
    return sub

forwards = build_df(is_forward, FORWARD_FEATURES, 'Forwards')
defense  = build_df(is_defense, DEFENSE_FEATURES, 'Defensemen')
goalies  = build_df(is_goalie,  GOALIE_FEATURES,  'Goalies')

## 8. Null Audit & Fill Strategy

In [ ]:
def null_audit(sub_df, features, label):
    feat_cols = [f for f in features if f in sub_df.columns]
    nulls = sub_df[feat_cols].isnull().mean().mul(100).round(1)
    nulls = nulls[nulls > 0].sort_values(ascending=False)
    if nulls.empty:
        print(f'{label}: No nulls in feature columns')
    else:
        print(f'{label} null rates (%):')
        print(nulls.to_string())
    print()

null_audit(forwards, FORWARD_FEATURES, 'Forwards')
null_audit(defense,  DEFENSE_FEATURES, 'Defensemen')
null_audit(goalies,  GOALIE_FEATURES,  'Goalies')

In [ ]:
# Fill remaining nulls:
# - Per-60 columns that are NaN because icetime = 0 → fill with 0
# - Percentage columns → fill with group median

def fill_nulls(sub_df, features):
    sub = sub_df.copy()
    feat_cols = [f for f in features if f in sub.columns]
    for col in feat_cols:
        if sub[col].isnull().any():
            if '_per60' in col or '_pct' in col or col in ['gsaa', 'gsaa_per60', 'gaa', 'sa_per60']:
                median_val = sub[col].median()
                sub[col] = sub[col].fillna(median_val)
            else:
                sub[col] = sub[col].fillna(0)
    return sub

forwards = fill_nulls(forwards, FORWARD_FEATURES)
defense  = fill_nulls(defense,  DEFENSE_FEATURES)
goalies  = fill_nulls(goalies,  GOALIE_FEATURES)

print('After fill:')
print(f'  Forwards nulls: {forwards[[f for f in FORWARD_FEATURES if f in forwards.columns]].isnull().sum().sum()}')
print(f'  Defense nulls:  {defense[[f for f in DEFENSE_FEATURES if f in defense.columns]].isnull().sum().sum()}')
print(f'  Goalies nulls:  {goalies[[f for f in GOALIE_FEATURES if f in goalies.columns]].isnull().sum().sum()}')

## 9. Spot Check — Top Earners

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.3f}'.format)

print('=== Top 10 Forwards by AAV ===')
top_f = forwards.nlargest(10, 'aav')[['player_name','start_year','aav','aav_pct_cap',
                                       'goals_per60','points_per60','ev_xGF_pct','pp_toi_share','gp_pct']]
print(top_f.to_string(index=False))

print()
print('=== Top 10 Defensemen by AAV ===')
top_d = defense.nlargest(10, 'aav')[['player_name','start_year','aav','aav_pct_cap',
                                      'primaryA_per60','ev_xGF_pct','pp_toi_share','blocks_per60','gp_pct']]
print(top_d.to_string(index=False))

print()
print('=== All Goalies by AAV ===')
top_g = goalies.nlargest(10, 'aav')[['player_name','start_year','aav','aav_pct_cap',
                                      'save_pct','hd_save_pct','gsaa','gaa','gp_pct']]
print(top_g.to_string(index=False))

## 10. Feature Correlation with Target

In [ ]:
def show_correlations(sub_df, features, label):
    feat_cols = [f for f in features if f in sub_df.columns]
    corr = sub_df[feat_cols + [TARGET]].corr()[TARGET].drop(TARGET)
    corr_abs = corr.abs().sort_values(ascending=False)
    print(f'{label} — top correlations with {TARGET}:')
    print(corr_abs.head(12).round(3).to_string())
    print()

show_correlations(forwards, FORWARD_FEATURES, 'Forwards')
show_correlations(defense,  DEFENSE_FEATURES, 'Defensemen')
show_correlations(goalies,  GOALIE_FEATURES,  'Goalies')

## 11. Export

In [ ]:
# Combine skaters into one file (forwards + defense)
# Add a position_group column so notebook 03 can split or use dummies
forwards['position_group'] = 'F'
defense['position_group']  = 'D'
goalies['position_group']  = 'G'

skaters = pd.concat([forwards, defense], ignore_index=True)

skaters_path = os.path.join(DATA_DIR, 'features_skaters.csv')
goalies_path = os.path.join(DATA_DIR, 'features_goalies.csv')

skaters.to_csv(skaters_path, index=False)
goalies.to_csv(goalies_path, index=False)

print(f'Skaters: {len(skaters):,} rows x {len(skaters.columns)} cols -> {skaters_path}')
print(f'Goalies: {len(goalies):,} rows x {len(goalies.columns)} cols -> {goalies_path}')
print()
print('Forward features:', FORWARD_FEATURES)
print()
print('Defense features:', DEFENSE_FEATURES)
print()
print('=' * 60)
print('STOP POINT — review before notebook 03')
print('  1. Top-correlating features make intuitive sense?')
print('  2. Spot-check values for known players look right?')
print('  3. No unexpected nulls remaining?')
print('  4. Goalie sample (37 rows) — model or exclude from main model?')
print('=' * 60)